# BERT-based Analysis of UberPeople.net Forum Data
This notebook performs advanced analysis of forum discussions to understand:
- Driver resistance strategies against algorithmic management
- Evolution of collective action patterns
- Platform-driver interaction dynamics
## Setup Instructions
Run the following cells to install required packages and initialize the environment.

In [4]:
# Package Installation
!pip install transformers torch pandas numpy bertopic umap-learn hdbscan plotly tqdm nltk seaborn openpyxl
# umap-learn: for dimensionality reduction
# hdbscan: for clustering

In [9]:
# Import Libraries

# Core data processing
import pandas as pd
import numpy as np
import torch  # For deep learning operations

# BERT and topic modeling
from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from umap import UMAP   # For dimensionality reduction
from hdbscan import HDBSCAN   # For clustering

# Visualization
import plotly.express as px
import seaborn as sns
import matplotlib.pyplot as plt

# Text processing
import nltk
from nltk.tokenize import sent_tokenize
import re

# Utility
import logging
import warnings
from tqdm.auto import tqdm    # For progress bars

# Mischellous
from sklearn.feature_extraction.text import CountVectorizer
import plotly.graph_objects as go
from google.colab import output
from datetime import datetime

# Configure settings
warnings.filterwarnings('ignore')   # Suppress warnings
nltk.download('punkt')              # Download tokenizer data
nltk.download('stopwords')          # Download stopwords
print("Available styles:", plt.style.available)
plt.style.use('seaborn-v0_8')

# plt.style.use('seaborn')
logging.basicConfig(level=logging.INFO)

# Enable dynamic output for Colab
output.enable_custom_widget_manager()

# Check GPU availability and set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Available styles: ['Solarize_Light2', '_classic_test_patch', '_mpl-gallery', '_mpl-gallery-nogrid', 'bmh', 'classic', 'dark_background', 'fast', 'fivethirtyeight', 'ggplot', 'grayscale', 'petroff10', 'seaborn-v0_8', 'seaborn-v0_8-bright', 'seaborn-v0_8-colorblind', 'seaborn-v0_8-dark', 'seaborn-v0_8-dark-palette', 'seaborn-v0_8-darkgrid', 'seaborn-v0_8-deep', 'seaborn-v0_8-muted', 'seaborn-v0_8-notebook', 'seaborn-v0_8-paper', 'seaborn-v0_8-pastel', 'seaborn-v0_8-poster', 'seaborn-v0_8-talk', 'seaborn-v0_8-ticks', 'seaborn-v0_8-white', 'seaborn-v0_8-whitegrid', 'tableau-colorblind10']
Using device: cpu


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
# Mount Google Drove
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Data Loading and Combination

import os
import pandas as pd

def load_uber_data(base_path='/content/drive/MyDrive/')